<a href="https://colab.research.google.com/github/kinjaljoshi/mech-interp-from-scratch/blob/master/1_sae_training_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sparse Autoencoders (SAEs): Intuition & Math

A **Sparse Autoencoder (SAE)** decomposes dense neural network activations (such as a 768-dimensional transformer embedding) into a larger set of interpretable, sparsely firing features.

### 1. The Forward Pass
Given an input activation vector $x \in \mathbb{R}^{d}$ (e.g., $d = 768$):

$$\text{features} = \text{ReLU}(W_{\text{enc}} x + b_{\text{enc}})$$

$$\hat{x} = W_{\text{dec}} (\text{features}) + b_{\text{dec}}$$

* **Encoder (`encoder(x)`):** Projects the dense input $x$ into a **higher-dimensional** latent space (for example, from `768` dimensions up to `4096` or `8192` features). Making the hidden layer wider than the input gives the model room to disentangle overlapping concepts (polysemanticity) into distinct directions.
* **Activation (`ReLU`):** Clamps all negative pre-activations to **exact zeros**, ensuring feature activations are strictly non-negative.
* **Decoder (`decoder(features)`):** Reconstructs the original input vector ($\hat{x}$) as a weighted linear combination of only the active (non-zero) features.

---

### 2. The Loss Function
The network is trained using a two-part objective:

$$\mathcal{L} = \underbrace{\|x - \hat{x}\|_2^2}_{\text{Reconstruction Error}} + \lambda \underbrace{\|\text{features}\|_1}_{\text{L1 Sparsity Penalty}}$$

* **Reconstruction Error ($\|x - \hat{x}\|_2^2$):** Forces the decoder output $\hat{x}$ to stay faithful to the original activation $x$, ensuring no important information is lost.
* **L1 Penalty ($\lambda \sum_i |\text{features}_i|$):** Penalizes the sum of all active feature magnitudes, scaled by a sparsity hyperparameter $\lambda$.
* **Why L1 produces exact zeros:** Unlike an L2 penalty (which shrinks large weights more than small ones, leaving many tiny non-zero values), the L1 penalty applies a **constant downward gradient** ($\lambda$) to every positive activation regardless of how small it is. Combined with `ReLU`, this pushes weak or irrelevant features all the way down to **exact `0.0`** for any given input.


In [ ]:
import torch, torch.nn as nn

enc = nn.Linear(3, 6)
dec = nn.Linear(6, 3)
x = torch.randn(2, 3)
rel = torch.relu(enc(x))
xhat = dec(rel)
print("x", x.shape, "-> features", rel.shape, "-> x_hat", xhat.shape)
print("dec.weight shape:", dec.weight.shape)

# --- 2. ReLU, L0 and L1 on a hand-made feature vector ---
z = torch.tensor([[-1., 2., 0., 3.]])    # pretend these are encoder scores
f = torch.relu(z)                        # -> [0, 2, 0, 3]
print("relu:", f)
print("L0 (how many are active):", (f >0).float().sum())
print("L1 (sum of magnitudes):", (f.abs().sum()))

# --- 3. The loss, by hand ---

x_true = torch.tensor([1., 2.])
y_hat = torch.tensor([0., 4.])

mse_loss = (x_true - y_hat).pow(2).mean().item()
print("mse_loss:", mse_loss)

f = torch.tensor([[0., 2., 0., 3.]])
l1_coeff = 0.1

l1 = l1_coeff * f.abs().sum(dim=1).item()

print("mse:", mse_loss, "| l1 term:", l1, "| total:", (mse_loss + l1))  # 2.5, 0.5, 3.0



x torch.Size([2, 3]) -> features torch.Size([2, 6]) -> x_hat torch.Size([2, 3])
dec.weight shape: torch.Size([3, 6])
relu: tensor([[0., 2., 0., 3.]])
L0 (how many are active): tensor(2.)
L1 (sum of magnitudes): tensor(5.)
mse_loss: 2.5
mse: 2.5 | l1 term: 0.5 | total: 3.0


In [ ]:
# --- 4. Why L1 and not L2: lets look at the gradients ---
f = torch.tensor([0.2, 5.], requires_grad=True)
f.abs().sum().backward()
print("L1 grad:", f.grad, "the same push toward 0 for both")
f.grad = None
(f ** 2).abs().sum().backward()
print("L2 grad:", f.grad, "pushes the big one harder, the small one barely")



L1 grad: tensor([1., 1.]) the same push toward 0 for both
L2 grad: tensor([ 0.4000, 10.0000]) pushes the big one harder, the small one barely


In [ ]:
# --- 5. Decoder column normalization ---
W = torch.tensor([[3., 1.],
                  [4., 0.]])                       # two feature directions (columns): [3,4] and [1,0]
print("column norms:", W.norm(dim=0))              # [5, 1]
W_unit = W / W.norm(dim=0, keepdim=True)           # divide each column by its length
print("normalized:\n", W_unit)                     # first column becomes [0.6, 0.8]
lin = nn.Linear(2, 2, bias=False)
with torch.no_grad():                              # edit weights in place without tracking gradients
    lin.weight.copy_(W_unit)                       # copy_ overwrites the stored weights
print("check norms after copy:", lin.weight.norm(dim=0))  # [1, 1]

column norms: tensor([5., 1.])
normalized:
 tensor([[0.6000, 1.0000],
        [0.8000, 0.0000]])
check norms after copy: tensor([1., 1.], grad_fn=<LinalgVectorNormBackward0>)


In [ ]:
!pip install transformer_lens

In [ ]:
from datasets import load_dataset
from transformer_lens.utilities import tokenize_and_concatenate
import torch
from transformer_lens import TransformerBridge

model = TransformerBridge.boot_transformers("gpt2")
TransformerBridge.enable_compatibility_mode(model)

# which layer to study (0-11); 6 is the middle read all
l_layers = model.cfg.n_layers
d = model.cfg.d_model

#hook for the residual stream going INTO blocks
#hook = [f"blocks.{i}.hook_resid_pre" for i in range(l_layers)]
hook = f"blocks.{6}.hook_resid_pre"


# pile-10k: 10,000 short text samples from a mixed web-text corpus
ds = load_dataset("NeelNanda/pile-10k", split="train")

# Tokenize and pack into fixed windows of 128 tokens, each starting with a BOS token

tok_ds = tokenize_and_concatenate(ds, model.tokenizer, max_length=128, add_bos_token=True)
all_tok = torch.tensor(tok_ds[:1000]["tokens"])
print(f"shape of all_tok- {all_tok.shape}")

# reading activations only, no gradientsat all layers
chunk = []
with torch.no_grad():
  for i in range(0, 1000, 50):
    _, cache = model.run_with_cache(all_tok[i:i+50].to(model.cfg.device), names_filter = lambda x : x in hook)
    chunk.append(cache[hook].cpu())
    if i % 200 == 0: print(f"done - {i}")

acts = torch.cat(chunk)
print(f"shape of acts- {acts.shape}")

# Check the BOS claim: how large is position 0 compared with the rest?
print("mean norm at BOS (position 0):", acts[:, 0].norm(dim=-1).mean().item())
print("mean norm at other positions: ", acts[:, 1:].norm(dim=-1).mean().item())


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

... (7 outputs left out)

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map (num_proc=10):   0%|          | 0/10000 [00:00<?, ? examples/s]

/tmp/ipykernel_30912/2983569348.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  all_tok = torch.tensor(tok_ds[:1000]["tokens"])


shape of all_tok- torch.Size([1000, 128])
done - 0
done - 200
done - 400
done - 600
done - 800
shape of acts- torch.Size([1000, 128, 768])
mean norm at BOS (position 0): 3026.098388671875
mean norm at other positions:  80.615234375


In [ ]:
#drop BOS activtions
acts = torch.cat(chunk)
acts = acts[:, 1:]
print(f"shape of acts- {acts.shape}")
#900 for traing and remaining for validation
train_acts = acts[:900].reshape(-1, d)
print("shape of train_acts", train_acts.shape)
test_acts = acts[900:].reshape(-1, d)
print("shape of test_act", test_acts.shape)
#
token_val = all_tok[:900, 1:]

shape of acts- torch.Size([1000, 127, 768])
shape of train_acts torch.Size([114300, 768])
shape of test_act torch.Size([12700, 768])


### Activation Norm Scaling (`scale`)

```python
scale = (d ** 0.5) / train_acts.norm(dim=-1).mean()
```

* **What it does:** Rescales every token activation vector so that the average vector length (L2 norm) across the dataset equals $\sqrt{d_{\text{model}}}$ ($\sqrt{768} \approx 27.71$ for GPT-2).
* **Why $\sqrt{d}$:** A $d$-dimensional vector whose coordinates each have unit variance (`1.0`) naturally has an expected Euclidean norm of $\sqrt{d}$.
(Before entering the attention or MLP blocks (and at the very end of the model), GPT-2 applies LayerNorm across the 768 features of every single token)
* **Why a single scalar:** Multiplying every vector by the same constant shrinks or stretches magnitudes uniformly without changing the angles (directions) between features.
* **Why it helps SAE training:** Transformer layers vary widely in raw activation magnitude. Normalizing to a standard $\sqrt{d}$ scale keeps the L1 sparsity penalty (`l1_coeff`) and learning rate consistent across different layers, and computing `scale` strictly on `train_acts` prevents data leakage.


In [ ]:
token_val = token_val.reshape(-1)
print(token_val.shape)
scale = d ** 0.5 / train_acts.norm(dim = -1).mean()
print(scale)
train_acts , test_acts = train_acts * scale, test_acts * scale
print(train_acts.shape, test_acts.shape)

torch.Size([114300])
tensor(0.3435)
torch.Size([114300, 768]) torch.Size([12700, 768])


## Sparse Autoencoder (SAE) Architecture & Training

An **SAE** decomposes GPT-2's dense `768`-dimensional residual stream into `6,144` sparse, interpretable feature directions (**8× expansion**).

### 1. Forward Pass Equations
* **Encoder:**
  $$f = \text{ReLU}\big((x - b_{\text{dec}}) W_{\text{enc}} + b_{\text{enc}}\big)$$
  * Centers the input by subtracting $b_{\text{dec}}$ before projecting `768` dimensions up to `6,144` latent features.
  * `ReLU` clamps negative pre-activations to exact `0.0`.
* **Decoder:**
  $$\hat{x} = f W_{\text{dec}} + b_{\text{dec}}$$
  * Reconstructs the original activation as a sparse weighted sum of learned directions, where **each row of $W_{\text{dec}}$** (`shape: [6144, 768]`) represents one feature's direction in GPT-2's activation space.

---

### 2. Loss Function
$$\mathcal{L} = \underbrace{\|x - \hat{x}\|_2^2}_{\text{Reconstruction Error}} + \lambda \underbrace{\sum_i f_i}_{\text{L1 Sparsity Penalty}}$$

* **Reconstruction error** pulls the model toward preserving all information from $x$.
* **L1 penalty (`l1_coeff` $\times \sum f_i$)** pushes weak feature activations down to exact `0.0` so only a handful of features fire per token.

---

### 3. Why Decoder Rows Have Unit Norm ($\|W_{\text{dec}}[i]\|_2 = 1$)
* **Preventing the scaling loophole:** In $f W_{\text{dec}}$, the model could cheat by shrinking $f$ (e.g., $\times 0.01$) to get a tiny L1 penalty while growing $W_{\text{dec}}$ ($\times 100$) to keep $\hat{x}$ identical.
* **Constraint:** Rescaling every row of $W_{\text{dec}}$ to unit L2 length (`1.0`) after each step forces the model to achieve low L1 loss through **genuine sparsity**.

---

### 4. Key Training Metrics to Monitor
* **L0 (`(f > 0).float().sum(-1).mean()`):** Average number of active (non-zero) features per token out of `6,144` (measures **sparsity**).
* **Fraction of Variance Unexplained (FVU):** $\frac{\text{MSE}(x, \hat{x})}{\text{Var}(x)}$ — proportion of activation variance missed by the reconstruction (measures **fidelity**; lower is better).


In [ ]:
import torch.nn as nn

d, n_feat = 768, 6144              # input width; number of SAE features (8x wider)
l1_coeff = 1.0                     # sparsity strength: higher = fewer active features, worse reconstruction
steps, batch= 2000, 512

class SAE(nn.Module):
    def __init__(self):
        super().__init__()
        W = torch.randn(n_feat, d)
        W = W / W.norm(dim=1, keepdim=True)              # random unit-length feature directions
        self.W_dec = nn.Parameter(W)                     # (n_feat, d): each ROW is a feature direction
        self.W_enc = nn.Parameter(W.T.clone())           # (d, n_feat): start as the transpose of the decoder
        self.b_enc = nn.Parameter(torch.zeros(n_feat))   # per-feature bias in the encoder
        self.b_dec = nn.Parameter(torch.zeros(d))        # shared offset: roughly the "average activation"

    def forward(self, x):
        f = torch.relu((x - self.b_dec) @ self.W_enc + self.b_enc)  # feature activations, mostly 0
        x_hat = f @ self.W_dec + self.b_dec                         # rebuild x from the active features
        return x_hat, f

    @torch.no_grad()                                     # editing weights directly, not a training step
    def normalize_decoder(self):
        self.W_dec /= self.W_dec.norm(dim=1, keepdim=True)  # snap every row back to length 1


In [ ]:
# Reference number: how much the activations vary around their mean (the "total variance")
total_var = ((test_acts - test_acts.mean(0)) ** 2).sum(-1).mean()


def evaluate(X):
    with torch.no_grad():                                # measuring only, no gradients
        x_hat, f = sae(X)
        mse = ((X - x_hat) ** 2).sum(-1).mean()          # squared error per token, summed over 768 dims
        return mse.item(), (mse / total_var).item(), (f > 0).sum(-1).float().mean().item()


sae = SAE()
sae.normalize_decoder()
opt = torch.optim.Adam(sae.parameters(), lr=4e-4)

for step in range(steps + 1):
    idx = torch.randint(0, len(train_acts), (batch,))    # pick 512 random training rows
    x = train_acts[idx]
    x_hat, f = sae(x)
    #reconstruction loss
    mse = ((x - x_hat) ** 2).sum(-1).mean()
    # sparsity term (sum over features, mean over batch)
    l1 = l1_coeff * f.abs().sum(-1).mean()
    loss = mse + l1
    opt.zero_grad()
    loss.backward()
    opt.step()
    sae.normalize_decoder()                              # keep decoder rows at unit length after every update
    if step % 250 == 0:
        _, fvu, l0 = evaluate(test_acts[:4000])           # check on held-out text, not training text
        print(f"step {step:4d} | train mse {mse.item():7.1f} | l1 {l1.item():7.1f} | "
              f"val frac. unexplained {fvu:.3f} | val L0 {l0:6.1f}")

step    0 | train mse  9701.0 | l1  2434.4 | val frac. unexplained 17.486 | val L0 2968.5
step  250 | train mse   167.3 | l1   189.7 | val frac. unexplained 0.396 | val L0  406.5
step  500 | train mse   137.8 | l1   137.5 | val frac. unexplained 0.335 | val L0  346.2
step  750 | train mse   107.6 | l1   109.7 | val frac. unexplained 0.287 | val L0  296.3
step 1000 | train mse    96.7 | l1    98.6 | val frac. unexplained 0.262 | val L0  254.6
step 1250 | train mse    87.0 | l1    89.5 | val frac. unexplained 0.248 | val L0  228.3
step 1500 | train mse    83.6 | l1    87.0 | val frac. unexplained 0.238 | val L0  212.2
step 1750 | train mse    76.0 | l1    80.9 | val frac. unexplained 0.233 | val L0  199.9
step 2000 | train mse    75.8 | l1    80.5 | val frac. unexplained 0.227 | val L0  191.1


In [ ]:
#checking feature stats
@torch.no_grad()
def feature_stats(X, bs = 2000):
  fires = torch.zeros(6144)
  for i in range(0, len(X), bs):
    _, f = sae(X[i:i+bs])
    fires += (f > 0).float().sum(0)
  return fires / len(X)   # firing rate per feature (fraction of tokens)

#Fraction of Variance Unexplained (FVU)
@torch.no_grad()
def fvu(X):
  x_hat, f = sae(X)
  mse = ((X - x_hat) ** 2).sum(-1).mean()
  var = (( X - X.mean(0)) ** 2).sum(-1).mean()
  return (mse / var).item(), (f > 0).float().sum(-1).mean().item()

In [ ]:
# Compare train and val on the SAME number of rows, each with its own variance
tr_sub = train_acts[:12700]
print("train FVU, L0:", fvu(tr_sub))
print("val   FVU, L0:", fvu(test_acts))


train FVU, L0: (0.19096136093139648, 179.33251953125)
val   FVU, L0: (0.22933638095855713, 212.1602325439453)


In [ ]:
# Dead and rare features, measured on 20,000 training tokens
rate = feature_stats(train_acts[:20000])
print("dead features (never fired):", (rate == 0).sum().item(), "of", n_feat)
print("fire on < 1 in 1000 tokens:", (rate < 0.001).sum().item())
print("fire on > 10% of tokens:  ", (rate > 0.10).sum().item())
print("mean firing rate:", rate.mean().item(), "| expected from L0:", 13.0 / n_feat)
print("rate quantiles (10/50/90/99%):", rate.quantile(torch.tensor([0.1, 0.5, 0.9, 0.99])).tolist())

dead features (never fired): 2 of 6144
fire on < 1 in 1000 tokens: 3
fire on > 10% of tokens:   133
mean firing rate: 0.029947109520435333 | expected from L0: 0.0021158854166666665
rate quantiles (10/50/90/99%): [0.01679999940097332, 0.024650000035762787, 0.039799999445676804, 0.14991208910942078]


In [ ]:
@torch.no_grad()                                      # measuring only, no gradients
def report(X, bs=2000):
    fires = torch.zeros(n_feat)                       # per feature: on how many tokens did it fire?
    active_pairs = 0.0                                # total (token, feature) pairs that were active
    for i in range(0, len(X), bs):                    # chunks of 2000 rows keep memory small
        _, f = sae(X[i:i+bs])
        on = (f > 0).float()                          # 1.0 where a feature is active, else 0.0
        fires += on.sum(0)                            # sum down the tokens -> per-feature counts
        active_pairs += on.sum().item()               # sum over everything -> total active pairs
    n = len(X)
    rate = fires / n                                  # firing rate per feature
    print(f"rows {n:6d} | mean L0 {active_pairs / n:6.1f} | sum of firing rates {rate.sum():6.1f} "
          f"| dead {(rate == 0).sum().item()} | rare (<1/1000) {(rate < 0.001).sum().item()}")
    return rate

rate_val = report(test_acts)                           # held-out rows
rate_tr = report(train_acts[:12700])                  # same train subset as before
rate_20k = report(train_acts[:20000])                 # the 20,000 rows the old stats used

rows  12700 | mean L0  212.2 | sum of firing rates  212.2 | dead 1 | rare (<1/1000) 3
rows  12700 | mean L0  179.3 | sum of firing rates  179.3 | dead 2 | rare (<1/1000) 3
rows  20000 | mean L0  184.0 | sum of firing rates  184.0 | dead 2 | rare (<1/1000) 3


**Results**

| Run | l1_coeff | Val FVU | L0 (val) |
|---|---|---|---|
| A | 5.0 | ~0.67 | ~13 |
| B | 1.0 | 0.229 (train 0.191) | 212 (train 179) |

- Run A reconstructs only about a third of the variance, so it is too sparse to read features from. We use Run B.
- Run B firing rates (first 20,000 train tokens): 2 of 6,144 features dead, 3 fire on fewer than 1 in 1,000 tokens, median feature fires on 2.5% of tokens (10th-90th percentile: 1.7%-4.0%), 133 features fire on more than 10% of tokens, and the 99th percentile is 15%.
- Consistency check: mean firing rate 0.02995 x 6,144 = 184 = mean L0 on the same 20,000 rows.


## 3.4 Reading SAE features

**Purpose.** Test whether the SAE's features are readable, and whether a feature's "best examples" give a fair picture of it. For six features (the 3 highest-firing and 3 random mid-rate ones) we check three things: how many distinct tokens drive its strongest activations (token detector vs. broader pattern), where in the 127-token window it fires (a position effect would show as a mean far from 63), and whether random weak activations read as clearly as the strongest ones.

**Setup so far (GPT-2 small, layer 6 residual stream, input to block 6)**
- Data: 1,000 windows x 127 tokens (BOS dropped). Train = first 900 windows (114,300 tokens), val = last 100 (12,700 tokens). Activations rescaled to mean norm sqrt(768) ~ 27.7 using train statistics only.
- SAE: 768 -> 6,144 features (8x), ReLU encoder, decoder rows kept at unit length, Adam lr 4e-4, batch 512, 2,000 steps, no L1 warm-up.



In [ ]:
from collections import Counter

S = 127                                              # token positions per window (128 minus the BOS we dropped)
tok_val = all_tok[900:, 1:]                          # (100, 127): token ids behind the held-out windows
tok_flat = tok_val.reshape(-1)                       # (12700,): one token id per val row, same order as val_acts

#test_acts
# ---------- 1. firing rate of every feature on val (computed here, so no dependency on earlier cells) ----------
with torch.no_grad():
  fires = torch.zeros(n_feat)
  for i in range(0, len(test_acts), 2000):
    _, f = sae(test_acts[i:i+2000])
    fires += (f > 0).float().sum(0)
rate_val = fires / len(test_acts)
print(f"sanity: sum of firing rates = {rate_val.sum():.1f} (should equal val mean L0, about 212)")



sanity: sum of firing rates = 212.2 (should equal val mean L0, about 212)


In [ ]:
# ---------- 2. choose features: 3 highest-firing + 3 random mid-rate ----------
high = rate_val.topk(3).indices.tolist()
print(high)
mid = ((rate_val > 0.005) & (rate_val < 0.05)).nonzero().squeeze(-1)     # features firing on 0.5%-5% of tokens
g = torch.Generator().manual_seed(0)
rand_mid = mid[torch.randperm(len(mid), generator=g)[:3]].tolist()
print(rand_mid)
picks = high + rand_mid

labels = ["HIGH-RATE"] * 3 + ["MID-RATE"] * 3

print("picked features:", picks)

[2793, 940, 4799]
[318, 5971, 3217]
picked features: [2793, 940, 4799, 318, 5971, 3217]


In [ ]:
# ---------- 3. activations of only these 6 features on validation input - test_acts ----------

with torch.no_grad():
  cols = []
  for i in range(0, len(test_acts), 2000):
    x = test_acts[i:i+2000]
    pre = (x - sae.b_dec) @ sae.W_enc[:, picks] + sae.b_enc[picks]
    cols.append(torch.relu(pre))
  Fp = torch.cat(cols)
  print(f"Fp shape - {Fp.shape}")


Fp shape - torch.Size([12700, 6])


In [ ]:
def context(r):
    # Row r of val_acts = window w, position p, because rows were flattened as (window, position)
    w, p = divmod(r, S)
    before = model.tokenizer.decode(tok_val[w, max(0, p - 10):p])         # up to 10 tokens of left context
    tok_ = model.tokenizer.decode(tok_val[w, p])                          # the token where the feature fires
    after = model.tokenizer.decode(tok_val[w, p + 1:p + 3])               # 2 tokens of right context
    return f"before - {before!r} token - [[{tok_}]] after - {after!r}"

# ---------- 4. per-feature report ----------
g2 = torch.Generator().manual_seed(1)                # separate seed for the random weak examples
for k, feat in enumerate(picks):
    col = Fp[:, k]                                   # this feature's activation on every val token
    active = (col > 0).nonzero().squeeze(-1)         # rows where it fires at all
    top50 = col.topk(50).indices                     # rows of its 50 strongest activations

    toks = tok_flat[top50].tolist()                  # the token at each of those rows
    top3 = [(model.tokenizer.decode(t), n) for t, n in Counter(toks).most_common(3)]
    print(f"\n=== {labels[k]} feature {feat} | fires on {len(active) / len(col):.1%} of val tokens | max act {col.max():.1f}")
    print(f"top-50: {len(set(toks))} distinct tokens | most common: {top3}")

    # Position in the window = row % S. If the feature ignored position, the mean would be near (S-1)/2 = 63.
    print(f"mean position  top-50: {(top50 % S).float().mean():.0f} | "
          f"all firing rows: {(active % S).float().mean():.0f} | uniform would be {(S - 1) / 2:.0f}")

    print("strongest 12:")
    vals, ix = col.topk(12)                          # the 12 strongest activations and their rows
    for v, r in zip(vals, ix):
        print(f"  {v:5.1f}  {context(r.item())}")

    print("6 random firing rows (not just the best ones):")
    pick_rows = active[torch.randperm(len(active), generator=g2)[:6]]
    for r in pick_rows:
        print(f"  {col[r]:5.1f}  {context(r.item())}")


=== HIGH-RATE feature 2793 | fires on 94.8% of val tokens | max act 12.3
top-50: 14 distinct tokens | most common: [('\n', 13), (' and', 7), (' ', 7)]
mean position  top-50: 71 | all firing rows: 66 | uniform would be 63
strongest 12:
   12.3  before - ' Mary University of London\nCategory:People from Berlin' token - [[<|endoftext|>]] after - 'MANILA'
   12.1  before - 'But almost wasn´t what I had in mind' token - [[<|endoftext|>]] after - '\n\n'
   11.7  before - 'iBold.ttf"\nend\n' token - [[<|endoftext|>]] after - 'Molly'
   11.6  before - ' needed to avoid “contamination”.' token - [[<|endoftext|>]] after - '1.'
   11.3  before - ' Like a Local\n\n### Travel with Children\n' token - [[
]] after - '### Regions'
   11.2  before - ' Children\n\n### Regions at a Glance\n' token - [[
]] after - '## On'
   11.1  before - ' his previously held second place position with 20 percent support' token - [[.]] after - ' While he'
   11.0  before - '-leaning registered voters August 12–18 – sho

## 3.4b Checking the readings

**Purpose.** Test four claims from 3.4 before trusting them: (A) how many distinct windows each feature's top-50 comes from (are the themes just a few documents?), (B) whether feature 4799 really depends on position, (C) whether feature 940 is driven by the `####` token itself or by position, and (D) whether feature 318 fires before tokens that continue the same word, compared with the base rate.

In [ ]:
pos = torch.arange(len(tok_flat)) % S            # position in the window (0..126) for every val row
win = torch.arange(len(tok_flat)) // S           # which window (0..99) each val row belongs to
col = lambda feat: Fp[:, picks.index(feat)]      # activation column of a picked feature, looked up by feature id

# --- A. How concentrated are the top examples? ---
for feat in picks:
    top50 = col(feat).topk(50).indices           # rows of the feature's 50 strongest activations
    print(f"feature {feat}: top-50 rows come from {win[top50].unique().numel()} of 100 windows")

# --- B. Feature 4799: mean activation by position bucket ---
edges = [0, 3, 6, 10, 20, 40, 80, 127]
c = col(4799)
print("\nfeature 4799, mean activation by position:")
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (pos >= lo) & (pos < hi)                 # rows whose position falls in this bucket
    print(f"  pos {lo:3d}-{hi - 1:3d}: mean act {c[m].mean():5.2f} | fires on {(c[m] > 0).float().mean():.0%}")

# --- C. Feature 940: '####' token identity vs position ---
ids = model.to_tokens("####", prepend_bos=False)[0].to("cpu")       # token id(s) for "####"
assert len(ids) == 1, "#### is not a single token here"
is_h = tok_flat == ids[0]                                  # True on every row where the token is ####
c = col(940)
print(f"\nfeature 940: '####' rows: {is_h.sum().item()} in {win[is_h].unique().numel()} windows")
print(f"  mean act on non-#### rows: {c[~is_h].mean():.2f} | on #### rows: {c[is_h].mean():.2f}")
for lo, hi in [(0, 40), (40, 90), (90, 127)]:
    m = is_h & (pos >= lo) & (pos < hi)                    # #### rows in this position range
    if m.sum() > 0:
        print(f"  #### at pos {lo}-{hi - 1}: n={m.sum().item():3d}, mean act {c[m].mean():.2f}")
r = torch.corrcoef(torch.stack([pos[is_h].float(), c[is_h]]))[0, 1]   # does activation rise with position?
print(f"  correlation of position with activation among #### rows: {r:.2f}")

# --- D. Feature 318: does the NEXT token continue the same word? ---
c = col(318)
def continues(r):
    t = model.tokenizer.decode(int(tok_flat[r + 1]))      # the token right after row r
    return len(t) > 0 and t[0].isalpha()                  # starts with a letter and no leading space
valid = (pos < S - 1).nonzero().squeeze(-1)                # rows that have a next token inside the same window
base = torch.tensor([continues(r) for r in valid.tolist()]).float().mean()   # base rate over all tokens
top = c.clone(); top[pos == S - 1] = -1                    # exclude the last position (no next token)
hit = torch.tensor([continues(r) for r in top.topk(100).indices.tolist()]).float().mean()
print(f"\nfeature 318: next token continues the word: {hit:.0%} of its top-100 vs {base:.0%} of all tokens")

feature 2793: top-50 rows come from 23 of 100 windows
feature 940: top-50 rows come from 28 of 100 windows
feature 4799: top-50 rows come from 23 of 100 windows
feature 318: top-50 rows come from 34 of 100 windows
feature 5971: top-50 rows come from 35 of 100 windows
feature 3217: top-50 rows come from 36 of 100 windows

feature 4799, mean activation by position:
  pos   0-  2: mean act 11.81 | fires on 100%
  pos   3-  5: mean act 11.85 | fires on 100%
  pos   6-  9: mean act 11.32 | fires on 100%
  pos  10- 19: mean act 10.11 | fires on 100%
  pos  20- 39: mean act  7.68 | fires on 100%
  pos  40- 79: mean act  3.38 | fires on 96%
  pos  80-126: mean act  0.52 | fires on 44%

feature 940: '####' rows: 542 in 32 windows
  mean act on non-#### rows: 2.65 | on #### rows: 4.51
  #### at pos 0-39: n=171, mean act 0.44
  #### at pos 40-89: n=216, mean act 4.09
  #### at pos 90-126: n=155, mean act 9.59
  correlation of position with activation among #### rows: 0.96

feature 318: next token

## 3.4 Results: reading SAE features (Run B, l1_coeff = 1.0)

Six features read: the 3 highest-firing and 3 random mid-rate ones, on 12,700 held-out tokens.

| Feature | Fires on | Finding |
|---|---|---|
| 4799 | 78% | Position feature. Mean act 11.8 at positions 0-5, falling to 0.5 at 80-126. |
| 940 | 81% | Not a `####` detector. Among `####` tokens, mean act rises 0.4 -> 4.1 -> 9.6 with position (r = 0.96). Cannot separate position from number of `####` already seen. |
| 2793 | 95% | Dense, not position-driven (mean position 66 vs 63 uniform). Unexplained. |
| 318 | 1.2% | Weak tendency to precede word continuations (29% of top-100 vs 16% base rate); post hoc, 71% of examples don't fit. |
| 5971 | 2.4% | No clear theme. |
| 3217 | 4.7% | Partly readable: ` &` and "Festivals & Events" in travel-guide place headings; the single top hit is an outlier. |

**Takeaways**
- The best examples overstate how readable a feature is: random weak activations read much worse than the top 12.
- At L0 around 200, many dense features track position or document structure, not concepts.
- Top examples come from 23-36 of 100 windows, so they are not a handful of passages, but windows can share a document.
- Limits: 6 features, 100 val windows, one SAE, one layer.